# Phase 1 - Arrondissement Data Exploration

Notebook d'exploration de la table `master_arrondissements` dans la voie pedagogique officielle du projet.


## 1. Setup & Imports


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.modeling import build_pedagogical_master_table

plt.style.use("seaborn-v0_8-whitegrid")


## 2. Data Loading


In [ ]:
master_arr = build_pedagogical_master_table()
master_arr.head()


## 3. Data Quality Check

Verification des types, des valeurs manquantes et de la cardinalite des 20 arrondissements.


In [ ]:
pd.DataFrame({
    "dtype": master_arr.drop(columns="geometry").dtypes.astype(str),
    "na_count": master_arr.drop(columns="geometry").isna().sum(),
})


In [ ]:
master_arr[["arrondissement_code", "arrondissement_name"]].sort_values("arrondissement_code")


## 4. Descriptive Statistics

Les variables `X1..X5` et la cible `Y` sont analysees a l'echelle arrondissement.


In [ ]:
analysis_columns = [
    "y_bin_count",
    "x1_population",
    "x2_commerce_restaurant_count",
    "x3_transport_station_count",
    "x4_green_area_m2",
    "x5_road_length_km",
]
master_arr[analysis_columns].describe().T


In [ ]:
plot_df = master_arr.sort_values("y_bin_count", ascending=False)
ax = plot_df.plot(
    x="arrondissement_name",
    y="y_bin_count",
    kind="bar",
    figsize=(12, 5),
    legend=False,
    color="#1f77b4",
)
ax.set_title("Observed street bins by arrondissement")
ax.set_xlabel("")
ax.set_ylabel("OSM waste basket count")
plt.xticks(rotation=75)
plt.show()


## 5. Geographic Visualization


In [ ]:
ax = master_arr.plot(
    column="y_bin_count",
    cmap="YlOrRd",
    figsize=(8, 8),
    legend=True,
    edgecolor="black",
    linewidth=0.5,
)
ax.set_title("Observed street bins by arrondissement")
ax.set_axis_off()
plt.show()


## 6. Correlation Analysis


In [ ]:
master_arr[analysis_columns].corr(numeric_only=True)


## 7. Key Findings & Next Steps

- La table primaire contient 20 observations, une par arrondissement.
- La cible `y_bin_count` et les variables `X1..X5` sont deja alignees avec la consigne pedagogique.
- L'etape suivante consiste a appliquer `KMeans`, creer `cl_2` et `cl_3`, puis ajuster la regression lineaire multiple.
